In [1]:
import numpy as np
import cv2
import serial
import time
from tensorflow.keras.models import load_model
import tensorflow as tf

In [ ]:
model = tf.keras.models.load_model('modelname.keras')

/Users/ajayyadav/Downloads/eAgro/eAgro/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 26 variables whereas the saved optimizer has 50 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
import cv2
import numpy as np
import serial
import time
import tensorflow as tf
import os
from datetime import datetime


SERIAL_PORT = '/dev/cu.SLAB_USBtoUART'  
BAUD_RATE = 115200

# Model path
MODEL_PATH = 'Model_name.keras'  

# Cooldown between sprays (seconds)
COOLDOWN_TIME = 10

# Confidence threshold
CONFIDENCE_THRESHOLD = 0.5


CAMERA_INDEX = 1  


IMG_HEIGHT = 256
IMG_WIDTH = 256


SAVE_CAPTURES = True
CAPTURE_DIR = "captured_images"


if SAVE_CAPTURES and not os.path.exists(CAPTURE_DIR):
    os.makedirs(CAPTURE_DIR)


print("Loading model...")
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check model path and try again.")
    exit(1)


class_names = [
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]


pesticide_map = {
    'Potato___healthy': 'WATER',
    'Tomato___healthy': 'WATER',
    
    'Potato___Early_blight': 'PEST1',
    'Potato___Late_blight': 'PEST1',
    'Tomato___Bacterial_spot': 'PEST1',
    'Tomato___Early_blight': 'PEST1',
    'Tomato___Late_blight': 'PEST1',
    'Tomato___Leaf_Mold': 'PEST1',
    'Tomato___Septoria_leaf_spot': 'PEST1',
    'Tomato___Target_Spot': 'PEST1',
    
    'Tomato___Spider_mites Two-spotted_spider_mite': 'PEST2',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'PEST2',
    'Tomato___Tomato_mosaic_virus': 'PEST2'
}


def preprocess_image(image):
    """
    Preprocess image for model prediction
    """

    image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    

    image = image / 255.0
    

    image = np.expand_dims(image, axis=0)
    
    return image


def enhance_image(image):

    image = cv2.convertScaleAbs(image, alpha=1.5, beta=0)

    image = cv2.GaussianBlur(image, (3, 3), 0)

    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    lab[:,:,0] = cv2.equalizeHist(lab[:,:,0])
    image = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return image


def predict_disease(image):
    """
    Predict disease from image
    """
    try:

        processed_image = preprocess_image(image)
        

        predictions = model.predict(processed_image, verbose=0)
        

        predicted_class = np.argmax(predictions[0])
        confidence = predictions[0][predicted_class]
        

        class_name = class_names[predicted_class]
        
        action = pesticide_map.get(class_name, 'UNKNOWN')
        
        return {
            'class_name': class_name,
            'confidence': float(confidence),
            'action': action,
            'prediction_array': predictions[0],
            'predicted_class_index': predicted_class
        }
    
    except Exception as e:
        print(f"Error in prediction: {e}")
        return None


def capture_and_process(cap, esp=None):
    """
    Capture a single image from camera and process it
    """
    print("\n📸 Capturing image...")
    
    # Capture frame
    ret, frame = cap.read()
    
    if not ret:
        print("❌ Error: Failed to capture image")
        return None
    

    if SAVE_CAPTURES:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{CAPTURE_DIR}/capture_{timestamp}.jpg"
        cv2.imwrite(filename, frame)
        print(f"💾 Image saved: {filename}")
    
    # Display captured image
    cv2.imshow("Captured Image", frame)
    cv2.waitKey(1000)  # Show for 1 second
    

    print("🔍 Analyzing image...")
    result = predict_disease(frame)
    
    if result is None:
        print("❌ Prediction failed")
        return None
    
    # Display results
    label = result['class_name']
    confidence = result['confidence']
    action = result['action']
    
    print("\n" + "="*50)
    print("📊 ANALYSIS RESULTS")
    print("="*50)
    
    if 'healthy' in label.lower():
        print(f"✅ STATUS: HEALTHY")
        print(f"🌿 Plant: {label.split('___')[1]}")
        color = "GREEN"
    else:
        print(f"⚠️ STATUS: DISEASE DETECTED")
        print(f"🦠 Disease: {label.split('___')[1]}")
        color = "RED"
    
    print(f"📈 Confidence: {confidence*100:.1f}%")
    print(f"💊 Action: {action}")
    

    print("\n📋 Top predictions:")
    predictions = result['prediction_array']
    top_indices = np.argsort(predictions)[-3:][::-1]
    for idx in top_indices:
        print(f"   - {class_names[idx]}: {predictions[idx]*100:.1f}%")
    
    print("="*50)
    
    # Send command to ESP32 if connected
    if esp and confidence > CONFIDENCE_THRESHOLD:
        print(f"\n🚀 Sending command to ESP32...")
        
        if action == 'WATER':
            send_command(esp, 'WATER')
        elif action == 'PEST1':
            send_command(esp, 'PEST1')
        elif action == 'PEST2':
            send_command(esp, 'PEST2')
        else:
            print("❓ Unknown action, no command sent")
    
    return result

# AUTO CAPTURE MODE 
def auto_capture_mode(cap, esp=None, interval=5):
    """
    Automatically capture and process images at regular intervals
    """
    print(f"\n🔄 Auto-capture mode enabled")
    print(f"📸 Capturing every {interval} seconds")
    print("Press 'q' to quit, 's' for manual capture\n")
    
    last_capture_time = 0
    
    while True:
        current_time = time.time()
        
        # Auto capture at interval
        if current_time - last_capture_time >= interval:
            print("\n" + "🕐"*10)
            print(f"Auto-capture triggered at {datetime.now().strftime('%H:%M:%S')}")
            
            result = capture_and_process(cap, esp)
            
            if result:
                last_capture_time = current_time
                
                # Apply cooldown for spraying
                if 'cooldown_until' in auto_capture_mode.__dict__:
                    if current_time < auto_capture_mode.cooldown_until:
                        wait_time = auto_capture_mode.cooldown_until - current_time
                        print(f"⏰ Spray cooldown: {wait_time:.1f} seconds remaining")
                else:
                    if result['action'] != 'WATER' and result['confidence'] > CONFIDENCE_THRESHOLD:
                        auto_capture_mode.cooldown_until = current_time + COOLDOWN_TIME
                        print(f"⏰ Cooldown active for {COOLDOWN_TIME} seconds")
        
        # Check for key press
        key = cv2.waitKey(100) & 0xFF
        
        if key == ord('q'):
            print("\n👋 Quitting auto-capture mode...")
            break
        elif key == ord('s'):
            print("\n📸 Manual capture requested")
            result = capture_and_process(cap, esp)
            if result and result['action'] != 'WATER' and result['confidence'] > CONFIDENCE_THRESHOLD:
                auto_capture_mode.cooldown_until = time.time() + COOLDOWN_TIME

#MANUAL MODE (Single Capture)
def manual_mode(cap, esp=None):
    """
    Manual mode - capture when user presses spacebar
    """
    print("\n📸 Manual capture mode")
    print("Press SPACEBAR to capture and analyze")
    print("Press 'q' to quit\n")
    
    while True:
        # Show live preview
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame")
            break
        
        # Add instructions to frame
        cv2.putText(frame, "Press SPACE to capture, 'q' to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        cv2.imshow("Live Preview - Press SPACE to Capture", frame)
        
        key = cv2.waitKey(30) & 0xFF
        
        if key == ord('q'):
            print("\n👋 Quitting...")
            break
        elif key == ord(' '):  # Spacebar
            print("\n" + "📸"*10)
            result = capture_and_process(cap, esp)
            
            # Wait for user to see results
            print("\nPress any key to continue...")
            cv2.waitKey(0)

#SERIAL COMMUNICATION
def send_command(esp, command):
    """
    Send command to ESP32
    """
    try:
 
        command_bytes = command.encode() + b'\n'
        esp.write(command_bytes)
        print(f"📤 Sent: {command}")
        
 
        time.sleep(0.5)
        if esp.in_waiting:
            response = esp.readline().decode().strip()
            print(f"📥 ESP response: {response}")
            
    except Exception as e:
        print(f"❌ Error sending command: {e}")


def test_camera():
    """
    Test camera connection
    """
    print(f"\nTesting camera (index {CAMERA_INDEX})...")
    cap = cv2.VideoCapture(CAMERA_INDEX)
    
    if not cap.isOpened():
        print(f"❌ Error: Could not open camera at index {CAMERA_INDEX}")
        print("Trying alternative camera indices...")
        

        for idx in [0, 2, 3]:
            print(f"Trying camera index {idx}...")
            cap = cv2.VideoCapture(idx)
            if cap.isOpened():
                print(f"✅ Found camera at index {idx}")
                print("Update CAMERA_INDEX in configuration to", idx)
                break
    
    if cap.isOpened():
        
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        ret, frame = cap.read()
        if ret:
            print(f"✅ Camera test successful!")
            print(f"Frame size: {frame.shape}")
            return cap
        else:
            print("❌ Camera opened but failed to capture frame")
    
    return None


def test_model():
    """
    Test if model is loaded correctly
    """
    print("\n🧪 Testing model...")
    print(f"Model type: {type(model)}")
    

    dummy_input = np.random.rand(1, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32)
    
    try:
        dummy_pred = model.predict(dummy_input, verbose=0)
        print(f"✅ Model test successful!")
        print(f"Output shape: {dummy_pred.shape}")
        print(f"Number of classes: {dummy_pred.shape[1]}")
        
        if dummy_pred.shape[1] != len(class_names):
            print(f"\n⚠️ WARNING: Model has {dummy_pred.shape[1]} classes,")
            print(f"but your class_names list has {len(class_names)} entries.")
            print("Please update class_names to match model output.")
        
    except Exception as e:
        print(f"❌ Model test failed: {e}")


def main():

    test_model()
    

    esp = None
    use_esp = input("\nConnect to ESP32? (y/n): ").lower() == 'y'
    
    if use_esp:
        print(f"\nOpening serial port {SERIAL_PORT} at {BAUD_RATE} baud...")
        try:
            esp = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
            time.sleep(2)  # Wait for Arduino to reset
            print("✅ Serial connection established!")
        except Exception as e:
            print(f"❌ Error opening serial port: {e}")
            print("Continuing without ESP32 connection...")
            esp = None
    

    cap = test_camera()
    
    if cap is None:
        print("❌ Could not open camera. Please check connection.")
        if esp:
            esp.close()
        return
    

    print("\n" + "="*50)
    print("SELECT OPERATION MODE")
    print("="*50)
    print("1. 📸 Single Capture Mode (Capture one image and process)")
    print("2. 🔄 Auto Capture Mode (Capture every X seconds)")
    print("3. 🎮 Manual Mode (Capture when you press spacebar)")
    print("="*50)
    
    mode = input("\nSelect mode (1/2/3): ").strip()
    
    if mode == '1':

        print("\n📸 Single Capture Mode")
        input("Press Enter to capture image...")
        result = capture_and_process(cap, esp)
        
        if result:
            print("\n✅ Processing complete!")

            ret, frame = cap.read()
            if ret:

                label = result['class_name']
                confidence = result['confidence']
                status = "HEALTHY" if 'healthy' in label.lower() else "DISEASE"
                color = (0, 255, 0) if status == "HEALTHY" else (0, 0, 255)
                
                cv2.putText(frame, f"Status: {status}", (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                cv2.putText(frame, f"Disease: {label.split('___')[1]}", (10, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, f"Confidence: {confidence*100:.1f}%", (10, 90), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, f"Action: {result['action']}", (10, 120), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                cv2.imshow("Final Result", frame)
                cv2.waitKey(3000)  
        
    elif mode == '2':

        try:
            interval = float(input("Enter capture interval (seconds) [default: 5]: ") or 5)
            auto_capture_mode(cap, esp, interval)
        except ValueError:
            print("Invalid interval, using 5 seconds")
            auto_capture_mode(cap, esp, 5)
            
    elif mode == '3':
      
        manual_mode(cap, esp)
        
    else:
        print("❌ Invalid mode selected")
    
    
    print("\n🧹 Cleaning up...")
    cap.release()
    if esp:
        esp.close()
    cv2.destroyAllWindows()
    print("✅ Done!")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n👋 Program interrupted by user")
        cv2.destroyAllWindows()
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        cv2.destroyAllWindows()

Loading model...
Model loaded successfully!

🧪 Testing model...
Model type: <class 'keras.src.models.sequential.Sequential'>
✅ Model test successful!
Output shape: (1, 13)
Number of classes: 13

Opening serial port /dev/cu.SLAB_USBtoUART at 115200 baud...
✅ Serial connection established!

Testing camera (index 1)...
✅ Camera test successful!
Frame size: (480, 640, 3)

SELECT OPERATION MODE
1. 📸 Single Capture Mode (Capture one image and process)
2. 🔄 Auto Capture Mode (Capture every X seconds)
3. 🎮 Manual Mode (Capture when you press spacebar)

📸 Single Capture Mode

📸 Capturing image...
💾 Image saved: captured_images/capture_20260421_155509.jpg
🔍 Analyzing image...

📊 ANALYSIS RESULTS
⚠️ STATUS: DISEASE DETECTED
🦠 Disease: Early_blight
📈 Confidence: 19.7%
💊 Action: PEST1

📋 Top predictions:
   - Tomato___Early_blight: 19.7%
   - Tomato___Bacterial_spot: 12.1%
   - Tomato___Late_blight: 11.0%

✅ Processing complete!

🧹 Cleaning up...
✅ Done!
